In [1]:
import polars as pl
from bert_score import BERTScorer
from tqdm.notebook import tqdm
from itertools import combinations

import numpy as np

In [2]:
df = pl.read_csv("full_corpus.csv")
df.shape

(388217, 199)

In [3]:
df.head()

argument,argument_id,prompt_id,raw_sequence_length,n_tokens,n_sentences,tokens_per_sentence,n_characters,avg_word_length,n_types,n_long_words,n_lemmas,n_VERB_VerbForm_Fin,n_VERB_VerbForm_Inf,n_VERB_VerbForm_Part,n_VERB_Mood_Imp,n_VERB_Mood_Ind,n_VERB_Mood_Sub,n_VERB_Tense_Past,n_VERB_Tense_Pres,n_VERB_Person_1,n_VERB_Person_2,n_VERB_Person_3,n_VERB_Number_Plur,n_VERB_Number_Sing,n_NOUN_Gender_Fem,n_NOUN_Gender_Masc,n_NOUN_Number_Plur,n_NOUN_Number_Sing,n_PRON_PronType_Dem,n_PRON_PronType_Int,n_PRON_PronType_Prs,n_PRON_PronType_Rel,n_PRON_Gender_Fem,n_PRON_Gender_Masc,n_PRON_Number_Plur,n_PRON_Number_Sing,…,n_entities,n_org,n_gpe,n_person,n_money,n_product,n_time,n_percent,n_work_of_art,n_quantity,n_norp,n_loc,n_event,n_ordinal,n_fac,n_law,n_language,question,stance,sociodemographic_info,sociodemographic_group,info,pro,contra,context,model,language,gender,age,education,civil_status,denomination,residence,political_spectrum,ID_question,topic,condition
str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,str,str,str,bool,str,str,str,str,str,str,str,str,str,i64,str,str
"""Die Begrenzung der Rente auf 1…","""2025-07-17T17:23:00.495917136Z""","""de_0""",-0.783945,-0.831965,-0.550714,0.237451,-0.773409,0.336463,-0.864157,1.148557,-0.910461,-1.481626,-0.70375,2.007832,-0.088682,-1.279825,-0.232455,-0.549852,-1.47543,-0.62804,-0.14845,-1.300616,-0.858245,-1.255052,1.108304,-1.104256,0.74926,0.284128,-0.491953,-0.077206,-0.616245,-0.697413,-0.428613,-0.577276,-0.587751,-1.046768,…,1.170055,-0.38352,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,"""Bei Ehepaaren ist die Höhe der…","""FAVOR""","""gender""","""Männlich""",null,null,null,false,"""llama_4_scout""","""de""","""Männlich""",null,null,null,null,null,null,32216,"""Welfare state & family""","""SD"""
"""Die Begrenzung der Rente für E…","""2025-07-17T17:23:01.835058734Z""","""de_0""",-0.699438,-0.761674,-0.469683,-0.26325,-0.687675,0.320903,-0.784716,1.114528,-0.781748,-0.63706,1.426023,0.315741,-0.088682,-0.415415,-0.232455,-0.549852,-0.674269,-0.62804,-0.14845,-0.373372,-0.858245,-0.360005,0.090713,-0.692199,0.884863,0.225443,-0.491953,-0.077206,-0.616245,1.480823,-0.428613,-0.577276,0.508981,-1.046768,…,1.37724,-0.38352,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""Bei Ehepaaren ist die Höhe der…","""FAVOR""","""gender""","""Männlich""",null,null,null,false,"""llama_4_scout""","""de""","""Männlich""",null,null,null,null,null,null,32216,"""Welfare state & family""","""SD"""
"""Die Begrenzung der Rente auf 1…","""2025-07-17T17:23:03.555754915Z""","""de_0""",-0.495087,-0.577161,-0.469683,0.545575,-0.484059,0.243636,-0.387508,0.2439,-0.459967,-0.292096,0.529169,-0.046674,-0.088682,-0.062346,-0.232455,-0.549852,-0.347034,-0.62804,-0.14845,0.005361,0.019654,-0.624737,1.159491,-1.710221,1.245836,0.2428,1.227492,-0.077206,0.494423,-0.697413,1.418575,-0.577276,-0.587751,0.895239,…,0.124402,-0.38352,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""Bei Ehepaaren ist die Höhe der…","""FAVOR""","""gender""","""Männlich""",null,null,null,false,"""llama_4_scout""","""de""","""Männlich""",null,null,null,null,null,null,32216,"""Welfare state & family""","""SD"""
"""Die Begrenzung der Rentenhöhe …","""2025-07-17T17:23:04.951396543Z""","""de_1""",-0.750142,-0.779247,-0.469683,-0.340281,-0.739473,0.213027,-0.758235,0.563675,-0.717392,0.277887,-0.816556,2.919519,-0.088682,0.521029,-0.232455,-0.549852,0.193655,-0.62804,-0.14845,0.631142,0.440315,-0.322711,0.654022,-0.649782,0.410251,-0.082656,-0.491953,-0.077206,-0.616245,1.571582,-0.428613,-0.577276,1.697108,-1.046768,…,0.450662,-0.38352,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""Bei Ehepaaren ist die Höhe der…","""FAVOR""","""age""","""18-34""",null,null,null,false,"""llama_4_scout""","""de""",null,"""18-34""",null,null,null,null,null,32216,"""Welfare state & family""","""SD"""
"""Ich denke, die Begrenzung der …","""2025-07-17

In [8]:
df["model"].unique()

model
str
"""llama_4_scout"""
"""occiglot-7b-eu5-instruct"""
"""llama_3.1-8b-instruct"""
"""gpt-4.1-mini"""
"""Human"""


In [9]:
# German

res = {
    "model": [],
    "prompt_id": [],
    "precision": [],
    "recall": [],
    "f1": []
}

scorer = BERTScorer(lang="de")
unique_combos = df.filter(pl.col("language")=="de", pl.col("model")!="Human").select("model", "prompt_id").unique()

for row in tqdm(unique_combos.iter_rows(), total=unique_combos.height):
    model, prompt_id = row

    sentences = df.filter(
        (pl.col("model") == model) & (pl.col("prompt_id") == prompt_id)
    )["argument"].to_list()

    pairs = list(combinations(sentences, 2))
    if not pairs:
        continue

    cands, refs = zip(*pairs)
    P, R, F1 = scorer.score(list(cands), list(refs))

    res["model"].append(model)
    res["prompt_id"].append(prompt_id)
    res["precision"].append(P.mean().item())
    res["recall"].append(R.mean().item())
    res["f1"].append(F1.mean().item())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  0%|          | 0/60634 [00:00<?, ?it/s]

In [11]:
res_df = pl.from_dict(res)

In [13]:
res_df.describe()

statistic,model,prompt_id,precision,recall,f1
str,str,str,f64,f64,f64
"""count""","""59123""","""59123""",59123.0,59123.0,59123.0
"""null_count""","""0""","""0""",0.0,0.0,0.0
"""mean""",null,null,0.74368,0.743742,0.743091
"""std""",null,null,0.079154,0.079053,0.078324
"""min""","""gpt-4.1-mini""","""de_0""",0.0,0.0,0.0
"""25%""",null,null,0.688769,0.688484,0.68675
"""50%""",null,null,0.753565,0.75374,0.753395
"""75%""",null,null,0.796441,0.796546,0.795657
"""max""","""occiglot-7b-eu5-instruct""","""de_9999""",1.0,1.0,1.0


In [14]:
res_df.write_csv("bert_score_de.csv")

In [24]:
# Italian

res = {
    "model": [],
    "prompt_id": [],
    "precision": [],
    "recall": [],
    "f1": []
}

scorer = BERTScorer(lang="it")
unique_combos = df.filter(pl.col("language")=="it", pl.col("model")!="Human").select("model", "prompt_id").unique()

skipped = 0

for row in tqdm(unique_combos.iter_rows(), total=unique_combos.height):
    model, prompt_id = row

    sentences = df.filter(
        (pl.col("model") == model), (pl.col("prompt_id") == prompt_id),
    )["argument"].to_list()

    pairs = list(combinations(sentences, 2))
    if not pairs:
        continue

    try:
        cands, refs = zip(*pairs)
        P, R, F1 = scorer.score(list(cands), list(refs))
    
        res["model"].append(model)
        res["prompt_id"].append(prompt_id)
        res["precision"].append(P.mean().item())
        res["recall"].append(R.mean().item())
        res["f1"].append(F1.mean().item())
    except AttributeError:
        print(f"Attribute error in row {row}")
        skipped += 1
        

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  0%|          | 0/21537 [00:00<?, ?it/s]

Attribute error in row ('occiglot-7b-eu5-instruct', 'it_1533')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_2272')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_3205')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_1085')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_2090')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_1044')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_3585')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_3996')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_5268')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_1151')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_1429')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_1203')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_2045')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_1784')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_5138')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it

Attribute error in row ('occiglot-7b-eu5-instruct', 'it_4387')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_4214')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_1191')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_3087')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_4763')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_2779')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_3854')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_1922')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_2227')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_3850')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_1086')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_5327')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_2486')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_3760')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_5372')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it

Attribute error in row ('occiglot-7b-eu5-instruct', 'it_4748')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_4157')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_1198')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_2111')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_4368')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_1144')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_1518')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_254')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_4370')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_1859')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_1650')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_4951')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_1390')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_639')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_2060')
Attribute error in row ('occiglot-7b-eu5-instruct', 'it_5

In [25]:
res_df = pl.from_dict(res)
res_df.describe()

statistic,model,prompt_id,precision,recall,f1
str,str,str,f64,f64,f64
"""count""","""20970""","""20970""",20970.0,20970.0,20970.0
"""null_count""","""0""","""0""",0.0,0.0,0.0
"""mean""",null,null,0.753305,0.753184,0.752665
"""std""",null,null,0.099519,0.099582,0.099083
"""min""","""gpt-4.1-mini""","""it_0""",0.0,0.0,0.0
"""25%""",null,null,0.666406,0.666507,0.66391
"""50%""",null,null,0.776832,0.777233,0.776946
"""75%""",null,null,0.825858,0.825689,0.82555
"""max""","""occiglot-7b-eu5-instruct""","""it_999""",1.0,1.0,1.0


In [26]:
skipped

354

In [27]:
res_df.write_csv("bert_score_it.csv")

In [28]:
# French

res = {
    "model": [],
    "prompt_id": [],
    "precision": [],
    "recall": [],
    "f1": []
}

scorer = BERTScorer(lang="fr")
unique_combos = df.filter(pl.col("language")=="fr", pl.col("model")!="Human").select("model", "prompt_id").unique()

skipped = 0

for row in tqdm(unique_combos.iter_rows(), total=unique_combos.height):
    model, prompt_id = row

    sentences = df.filter(
        (pl.col("model") == model), (pl.col("prompt_id") == prompt_id),
    )["argument"].to_list()

    pairs = list(combinations(sentences, 2))
    if not pairs:
        continue

    try:
        cands, refs = zip(*pairs)
        P, R, F1 = scorer.score(list(cands), list(refs))
    
        res["model"].append(model)
        res["prompt_id"].append(prompt_id)
        res["precision"].append(P.mean().item())
        res["recall"].append(R.mean().item())
        res["f1"].append(F1.mean().item())
    except AttributeError:
        print(f"Attribute error in row {row}")
        skipped += 1
        

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  0%|          | 0/45927 [00:00<?, ?it/s]

In [29]:
skipped

0

In [30]:
res_df = pl.from_dict(res)
res_df.describe()

statistic,model,prompt_id,precision,recall,f1
str,str,str,f64,f64,f64
"""count""","""45800""","""45800""",45800.0,45800.0,45800.0
"""null_count""","""0""","""0""",0.0,0.0,0.0
"""mean""",null,null,0.767484,0.767416,0.767013
"""std""",null,null,0.079951,0.079833,0.079369
"""min""","""gpt-4.1-mini""","""fr_0""",0.386301,0.373209,0.389852
"""25%""",null,null,0.722815,0.72326,0.72322
"""50%""",null,null,0.779975,0.779787,0.778821
"""75%""",null,null,0.822311,0.822063,0.821652
"""max""","""occiglot-7b-eu5-instruct""","""fr_9999""",1.0,1.0,1.0


In [31]:
res_df.write_csv("bert_score_fr.csv")